In [77]:
# Verification cell: just run it (Shift+Enter). Provided for you.
# importlib.util is a submodule: importing the parent alone does not bind it,
# and this cell now runs before anything else has imported it for us.
import importlib.util

REQUIRED = [
    "numpy", "pandas", "matplotlib", "seaborn", "sklearn", "openpyxl",
]
missing = [name for name in REQUIRED if importlib.util.find_spec(name) is None]

if missing:
    print("MISSING packages:", ", ".join(missing))
    print("-> Please install the missing packages using pip install <package-name>")
else:
    print("Environment OK.\nEvery package the project needs is installed!")

Environment OK.
Every package the project needs is installed!


In [78]:
# Import all packages needed for the project
# Unused packages can be safely removed from this cell
import sys
from functools import partial
from pathlib import Path

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import numpy as np
# import matplotlib.pyplot as plt
# import seaborn as sns
# import sklearn as sk
# import openpyxl as op

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
FIGSIZE = (10, 6)



In [79]:
# Import data that is gonna be used for this project

RAW_TRAIN_DATA_PATH = "../data/train.csv"
CLIENT_DATA_PATH = "../data/clientes.csv"
raw_train_data = pd.read_csv(RAW_TRAIN_DATA_PATH, index_col="ID", float_precision='round_trip')
client_data = pd.read_csv(CLIENT_DATA_PATH, index_col="customer_id", float_precision='round_trip')

In [80]:
print(f"Raw train data: {raw_train_data.shape[0]:,} rows x {raw_train_data.shape[1]} feature columns, indexed by ID")
raw_train_data.head()

Raw train data: 6,534 rows x 38 feature columns, indexed by ID


,Total a Pagar,Total Bruto,Total Liquido,Total Imp_,Total Portes,Total Desc_ Global,Total Desc_ Linha,Custo,N_ Detalhes,invoice_count_current_purchase,...,spend_365d,customer_id,payment_type,salesperson,commercial_zone,transport_method,warehouse,document_series,purchase_date,random_noise
ID,,,,,,,,,,,,,,,,,,,,,
10000000,558.11,510.90,453.75,104.36,0.0,0.0,57.15,153.099911,4.0,1.0,...,858.30,660784,Maybe Tomorrow Payment 10,VLOOKUP Master 9,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 61,Final Excel Warehouse 1,Ctrl Z Series 2,2024-10-29,-0.660448
10000001,978.80,830.02,795.77,183.03,0.0,0.0,34.25,472.943134,32.0,1.0,...,2500.68,189146,Maybe Tomorrow Payment 7,VLOOKUP Master 1,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 36,Final Excel Warehouse 1,Ctrl Z Series 1,2017-07-08,0.924510
10000002,10.33,12.00,8.40,1.93,0.0,0.0,3.60,6.097485,1.0,1.0,...,7899.80,103940,Maybe Tomorrow Payment 9,VLOOKUP Master 1,Decorative Dashboard Zone 4,NaN,Final Excel Warehouse 1,Ctrl Z Series 3,2019-03-19,-0.258730
10000003,856.83,696.61,696.61,160.22,0.0,0.0,0.00,408.167275,9.0,1.0,...,270.33,512927,Maybe Tomorrow Payment 7,VLOOKUP Master 4,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 71,Final Excel Warehouse 1,Ctrl Z Series 3,2018-11-10,1.448402
10000004,1616.10,1488.85,1313.90,302.20,0.0,0.0,174.95,485.401864,33.0,1.0,...,2062.50,122356,Maybe Tomorrow Payment 12,NaN,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 61,Final Excel Warehouse 1,Ctrl Z Series 2,2021-11-30,-1.058890


In [81]:
print(f"Clients data: {client_data.shape[0]:,} rows x {client_data.shape[1]} feature columns, indexed by ID")
client_data.head()

Clients data: 1,143 rows x 12 feature columns, indexed by ID


,country_or_region,commercial_region,island,municipality,city,postal_area,postal_code,street,address,customer_type,company_name,customer_since_year
customer_id,,,,,,,,,,,,
842570,Country CT11: Land of The Final Version,Region RG37: We Will See Later,NaN,NaN,City CY417: Missing Cell Port,NaN,Postal code PC828: #REF! Delivery Route,NaN,Address AD1120: Cell Without a Home,NaN,Excel Is Not A Database Ltd 150,2018
770941,Country CT9: Land of The Final Version,NaN,NaN,NaN,NaN,NaN,Postal code PC56: #REF! Delivery Route,NaN,Address AD62: Cell Without a Home,NaN,Excel Is Not A Database Ltd 1064,2021
827053,Country CT10: Land of The Final Version,Region RG17: We Will See Later,Island IS2: The Pivot Archipelago,Municipality MU14: Council of Last Minute Edits,NaN,Postal area PA271: Hidden Sheet Sector,Postal code PC827: #REF! Delivery Route,Street ST170: PowerPoint New Walk,Address AD449: Cell Without a Home,NaN,Excel Is Not A Database Ltd 113,2016
494648,Country CT10: Land of The Final Version,Region RG17: We Will See Later,Island IS2: The Pivot Archipelago,Municipality MU14: Council of Last Minute Edits,NaN,Postal area PA271: Hidden Sheet Sector,Postal code PC826: #REF! Delivery Route,NaN,Address AD454: Cell Without a Home,NaN,Excel Is Not A Database Ltd 430,2015
429291,Country CT10: Land of The Final Version,Region RG17: We Will See Later,Island IS2: The Pivot Archipelago,Municipality MU14: Council of Last Minute Edits,NaN,Postal area PA271: Hidden Sheet Sector,Postal code PC826: #REF! Delivery Route,NaN,Address AD453: Cell Without a Home,NaN,Excel Is Not A Database Ltd 673,2015


In [82]:
def out_of_domain(frame, cols, errors):
    """Return rows with at least one known domain violation in `cols`."""
    bad = pd.Series(False, index=frame.index)
    for column in cols:
        bad |= errors[column](frame[column])
    return bad


# These rules describe impossible values for purchase records; missing values
# are reported separately because they may need imputation rather than removal.
train_data_domain_errors = {
    "Total a Pagar": lambda values: values.notna() & (values <= 0),
    "Total Bruto": lambda values: values.notna() & (values <= 0),
    "Total Liquido": lambda values: values.notna() & (values < 0),
    "Total Imp_": lambda values: values.notna() & (values < 0),
    "Total Portes": lambda values: values.notna() & (values < 0),
    "Total Desc_ Global": lambda values: values.notna() & (values < 0),
    "Total Desc_ Linha": lambda values: values.notna() & (values < 0),
    "Custo": lambda values: values.notna() & (values < 0),
    "N_ Detalhes": lambda values: values.notna() & ((values < 1) | (values % 1 != 0)),
    "invoice_count_current_purchase": lambda values: values.notna() & ((values < 1) | (values % 1 != 0)),
    "repeat_purchase_90d": lambda values: values.notna() & ~values.isin([0, 1]),
    "days_since_previous_purchase": lambda values: values.notna() & (values < 0),
    "customer_tenure_days": lambda values: values.notna() & (values < 0),
    "prior_purchase_count": lambda values: values.notna() & ((values < 0) | (values % 1 != 0)),
    "prior_total_spend": lambda values: values.notna() & (values < 0),
    "prior_mean_purchase_value": lambda values: values.notna() & (values < 0),
    "prior_std_purchase_value": lambda values: values.notna() & (values < 0),
    "prior_min_purchase_value": lambda values: values.notna() & (values < 0),
    "prior_max_purchase_value": lambda values: values.notna() & (values < 0),
    "prior_mean_gap_days": lambda values: values.notna() & (values < 0),
    "prior_std_gap_days": lambda values: values.notna() & (values < 0),
    "purchase_count_30d": lambda values: values.notna() & ((values < 0) | (values % 1 != 0)),
    "spend_30d": lambda values: values.notna() & (values < 0),
    "purchase_count_90d": lambda values: values.notna() & ((values < 0) | (values % 1 != 0)),
    "spend_90d": lambda values: values.notna() & (values < 0),
    "purchase_count_180d": lambda values: values.notna() & ((values < 0) | (values % 1 != 0)),
    "spend_180d": lambda values: values.notna() & (values < 0),
    "purchase_count_365d": lambda values: values.notna() & ((values < 0) | (values % 1 != 0)),
    "spend_365d": lambda values: values.notna() & (values < 0),
    "customer_id": lambda values: values.notna() & ((values <= 0) | (values % 1 != 0)),
}

numeric_columns = set(train_data_domain_errors) | {"random_noise"}
column_report = []
invalid_by_column = {}
for column in raw_train_data.columns:
    values = raw_train_data[column]
    invalid = pd.Series(False, index=raw_train_data.index)

    if column in numeric_columns:
        if pd.api.types.is_numeric_dtype(values):
            invalid |= values.notna() & ~np.isfinite(values)
            if column in train_data_domain_errors:
                invalid |= train_data_domain_errors[column](values)
        else:
            invalid |= values.notna()
    elif column == "purchase_date":
        parsed_dates = pd.to_datetime(values, errors="coerce")
        invalid |= values.notna() & parsed_dates.isna()
    else:
        invalid |= values.notna() & ~values.map(lambda value: isinstance(value, str))
        invalid |= values.map(lambda value: isinstance(value, str) and not value.strip())

    invalid_by_column[column] = invalid
    examples = [
        f"ID {row_id}: {value!r}"
        for row_id, value in values[invalid].head(3).items()
    ]
    column_report.append({
        "column": column,
        "missing": int(values.isna().sum()),
        "invalid": int(invalid.sum()),
        "invalid_examples": "; ".join(examples),
    })

quality_report = pd.DataFrame(column_report).set_index("column")
invalid_rows = out_of_domain(raw_train_data, list(train_data_domain_errors), train_data_domain_errors)
for invalid in invalid_by_column.values():
    invalid_rows |= invalid
print(f"Checked {len(raw_train_data.columns)} columns across {len(raw_train_data):,} rows.")
row_ids = pd.to_numeric(pd.Series(raw_train_data.index), errors="coerce")
invalid_ids = row_ids.isna() | (row_ids <= 0) | (row_ids % 1 != 0)
print(f"Rows with at least one invalid value: {invalid_rows.sum():,}")
print(f"Invalid row IDs: {invalid_ids.sum():,}; duplicate row IDs: {raw_train_data.index.has_duplicates}")
display(quality_report)

Checked 38 columns across 6,534 rows.
Rows with at least one invalid value: 63
Invalid row IDs: 0; duplicate row IDs: False


,missing,invalid,invalid_examples
column,,,
Total a Pagar,65,0,
Total Bruto,65,0,
Total Liquido,65,0,
Total Imp_,65,0,
Total Portes,65,0,
Total Desc_ Global,65,0,
Total Desc_ Linha,65,0,
Custo,65,0,
N_ Detalhes,65,0,


In [83]:
# Drop only rows identified as invalid by the checks above; retain missing values.
train_processed = raw_train_data.loc[~invalid_rows].copy()

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)
processed_path = processed_dir / "train_processed.csv"
train_processed.to_csv(processed_path, index=True, index_label=raw_train_data.index.name)

print(f"Removed {invalid_rows.sum():,} invalid rows; retained {len(train_processed):,} of {len(raw_train_data):,} rows.")
print(f"Saved processed training data to {processed_path.resolve()}")

Removed 63 invalid rows; retained 6,471 of 6,534 rows.
Saved processed training data to /home/yoru/Documents/novaims/Y1/S1/ML/ML_Project/data/processed/train_processed.csv


In [84]:
TRAIN_DATA_PATH = "../data/processed/train_processed.csv"

train_data = pd.read_csv(TRAIN_DATA_PATH, index_col='ID', float_precision='round_trip')

In [85]:
train_data.info()

<class 'pandas.DataFrame'>
Index: 6471 entries, 10000000 to 10007674
Data columns (total 38 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Total a Pagar                   6406 non-null   float64
 1   Total Bruto                     6406 non-null   float64
 2   Total Liquido                   6406 non-null   float64
 3   Total Imp_                      6406 non-null   float64
 4   Total Portes                    6406 non-null   float64
 5   Total Desc_ Global              6406 non-null   float64
 6   Total Desc_ Linha               6406 non-null   float64
 7   Custo                           6406 non-null   float64
 8   N_ Detalhes                     6406 non-null   float64
 9   invoice_count_current_purchase  6406 non-null   float64
 10  repeat_purchase_90d             6471 non-null   int64  
 11  days_since_previous_purchase    5613 non-null   float64
 12  customer_tenure_days            6406 no